<a href="https://colab.research.google.com/github/Martelletti27/fiap_cursotiao_pbl/blob/main/Fase%206/Parte%202/Felipe_rm567521_pbl_fase6_Yolo5n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FASE 6 — ENTREGA 2
## FarmTech Solutions | Comparação de Abordagens de Visão Computacional

Este notebook implementa e compara três abordagens de detecção/classificação
de imagens utilizando o mesmo dataset da Entrega 1 (animais e maquinários):

1. **YOLOv8 adaptável** — já realizado na Entrega 1 (resultados resgatados para comparação)
2. **YOLOv5 tradicional** — abordagem padrão apresentada no Capítulo 3 da disciplina
3. **CNN treinada do zero** — rede convolucional construída manualmente com Keras/TensorFlow

O objetivo é avaliar criticamente cada abordagem em termos de:
- Facilidade de uso e integração
- Precisão do modelo
- Tempo de treinamento
- Tempo de inferência

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Abordagem 1: YOLOv5 Tradicional

O Capítulo 3 da disciplina apresenta o **YOLOv5** como a abordagem padrão
de detecção de objetos com YOLO. Diferente do YOLOv8 utilizado na Entrega 1,
que possui uma API de alto nível (`ultralytics`), o YOLOv5 é executado via
scripts Python clonados diretamente do repositório oficial do GitHub
(Código-fonte 3 do capítulo).

Isso exige etapas mais manuais de configuração: clone do repositório,
instalação de dependências e chamada de scripts via linha de comando,
tornando o processo mais transparente, porém menos ágil.

Utilizaremos o **mesmo dataset e o mesmo `dataset.yaml`** da Entrega 1,
garantindo uma comparação justa entre as abordagens.

### Estrutura do dataset no Google Drive:

In [ ]:
# Passo 1: Clone do repositório YOLOv5 (conforme Código-fonte 3 do capítulo)
!git clone https://github.com/ultralytics/yolov5.git

# Passo 2: Instalação automática dos requerimentos (conforme Código-fonte 4 do capítulo)
!pip install -r yolov5/requirements.txt -q

print("Repositório YOLOv5 clonado e dependências instaladas com sucesso!")

In [ ]:
import yaml

dataset_yaml_path = '/content/drive/MyDrive/Dataset/dataset.yaml'

with open(dataset_yaml_path, 'r') as f:
    conteudo = yaml.safe_load(f)

print("Conteúdo atual do dataset.yaml:")
print(yaml.dump(conteudo, sort_keys=False, allow_unicode=True))

### Atualização do dataset.yaml para o YOLOv5

O arquivo `dataset.yaml` foi criado na Entrega 1 com caminhos apontando
para as pastas do Drive. Antes de treinar com o YOLOv5, vamos garantir
que os caminhos estejam corretos e no formato esperado pelo YOLOv5,
conforme o Código-fonte 5 do Capítulo 3.

O formato esperado é:
```yaml
train: /content/drive/MyDrive/Dataset/images/train
val:   /content/drive/MyDrive/Dataset/images/val
nc: 2
names:
  0: maquinarios
  1: animais
```

In [ ]:
import yaml
import os

dataset_yaml_path = '/content/drive/MyDrive/Dataset/dataset.yaml'

# Atualiza o yaml garantindo os caminhos corretos para o YOLOv5
data_config = {
    'train': '/content/drive/MyDrive/Dataset/images/train',
    'val':   '/content/drive/MyDrive/Dataset/images/val',
    'nc': 2,
    'names': {0: 'maquinarios', 1: 'animais'}
}

with open(dataset_yaml_path, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False, allow_unicode=True)

print("dataset.yaml atualizado com sucesso!")
print(yaml.dump(data_config, sort_keys=False, allow_unicode=True))

In [ ]:
import os

# Clona o repositório YOLOv5 apenas se ainda não existir
if not os.path.exists('yolov5'):
    !git clone https://github.com/ultralytics/yolov5.git
    print("Repositório YOLOv5 clonado com sucesso!")
else:
    print("Repositório YOLOv5 já existe, pulando o clone.")

# Instalação das dependências (sempre executa para garantir que está tudo instalado)
!pip install -r yolov5/requirements.txt -q

print("Dependências instaladas com sucesso!")

### Treinamento do YOLOv5

Seguindo o Código-fonte 7 do Capítulo 3, executamos o treinamento com o
comando `python yolov5/train.py`, passando os seguintes argumentos:

- `--data`: caminho para o `dataset.yaml`
- `--weights yolov5s.pt`: pesos pré-treinados do modelo small (mais leve)
- `--img 640`: tamanho de entrada das imagens
- `--epochs 50`: número de épocas (usamos 50 para comparar com as 30 e 60 da Entrega 1)

O modelo treinado será salvo em `yolov5/runs/train/fazenda_yolov5/weights/best.pt`.

In [ ]:
import time
import os

dataset_yaml_path = '/content/drive/MyDrive/Dataset/dataset.yaml'

# Marca início do treinamento
inicio_treino_v5 = time.time()

# Código-fonte 7 do capítulo: treinamento do YOLOv5
!python yolov5/train.py \
    --data {dataset_yaml_path} \
    --weights yolov5s.pt \
    --img 640 \
    --epochs 50 \
    --name fazenda_yolov5 \
    --exist-ok

fim_treino_v5 = time.time()
tempo_treino_v5 = fim_treino_v5 - inicio_treino_v5

print(f"\nTreinamento YOLOv5 concluído!")
print(f"Tempo total de treinamento: {tempo_treino_v5:.2f} segundos ({tempo_treino_v5/60:.2f} minutos)")

In [ ]:
from IPython.display import Image, display
import os

results_dir = 'yolov5/runs/train/fazenda_yolov5'

# Gráfico geral de métricas (loss, precision, recall, mAP)
results_png = os.path.join(results_dir, 'results.png')
if os.path.exists(results_png):
    print("Gráfico de métricas:")
    display(Image(filename=results_png))

# Matriz de confusão
confusion_path = os.path.join(results_dir, 'confusion_matrix.png')
if os.path.exists(confusion_path):
    print("Matriz de Confusão:")
    display(Image(filename=confusion_path))

# Exemplos de validação com bounding boxes
for val_img in ['val_batch0_pred.jpg', 'val_batch1_pred.jpg']:
    val_path = os.path.join(results_dir, val_img)
    if os.path.exists(val_path):
        print(f"Predições na validação ({val_img}):")
        display(Image(filename=val_path))

In [ ]:
# Localiza o melhor modelo treinado
best_weights_v5 = 'yolov5/runs/train/fazenda_yolov5/weights/best.pt'

if os.path.exists(best_weights_v5):
    print(f"Modelo encontrado em: {best_weights_v5}")
else:
    print("Modelo não encontrado. Verifique se o treinamento foi concluído.")

### Validação do Modelo YOLOv5

Após o treinamento, avaliamos o modelo no conjunto de validação.
Conforme o Código-fonte 8 do Capítulo 3, ao final do treino o YOLOv5
já exibe automaticamente uma tabela com as métricas por classe:

| Classe      | Imagens | Instâncias | Precision | Recall | mAP50 | mAP50-95 |
|-------------|---------|------------|-----------|--------|-------|----------|
| all         |20       |86          |0.818         |0.847      |0.859     |0.523       |
| maquinarios |20       |12          |1.000         |0.883     |0.916     |0.585      |
| animais     |20       |74          |0.635         |0.811      |0.801     |0.462       |

As métricas avaliadas são:
- **Precision (P)**: proporção de detecções corretas entre todas as detecções feitas
- **Recall (R)**: proporção de objetos reais corretamente detectados
- **mAP50**: precisão média com limiar de IoU = 0.5
- **mAP50-95**: precisão média variando IoU de 0.5 a 0.95 (métrica mais exigente)

In [ ]:
# Validação explícita do modelo (val.py)
print("Iniciando validação YOLOv5...")

!python yolov5/val.py \
    --weights {best_weights_v5} \
    --data {dataset_yaml_path} \
    --img 640

print("Validação YOLOv5 concluída.")

### Teste do Modelo YOLOv5

Seguindo o Código-fonte 10 do Capítulo 3, realizamos o teste com as
imagens reservadas para essa etapa (`images/test`), que **não foram vistas**
durante o treinamento nem a validação.

O script `detect.py` processa cada imagem e salva os resultados com as
bounding boxes desenhadas em `yolov5/runs/detect/teste_yolov5/`.

Também medimos o **tempo de inferência** por imagem para comparação
com as outras abordagens.

In [ ]:
import subprocess

test_images_path = '/content/drive/MyDrive/Dataset/images/test'

# Conta imagens de teste
imagens_teste = [
    f for f in os.listdir(test_images_path)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
]
n_imagens_teste = len(imagens_teste)
print(f"Imagens de teste encontradas: {n_imagens_teste}")

# Marca início da inferência
inicio_inferencia_v5 = time.time()

# Código-fonte 10 do capítulo: teste independente do modelo
result = subprocess.run(
    f'python yolov5/detect.py '
    f'--weights {best_weights_v5} '
    f'--img 640 '
    f'--source "{test_images_path}" '
    f'--data {dataset_yaml_path} '
    f'--name teste_yolov5 '
    f'--exist-ok',
    shell=True,
    capture_output=True,
    text=True
)

fim_inferencia_v5 = time.time()
tempo_inferencia_v5 = fim_inferencia_v5 - inicio_inferencia_v5

print(result.stdout)
print(result.stderr)

print(f"\nTeste YOLOv5 concluído!")
print(f"Tempo total de inferência: {tempo_inferencia_v5:.4f} segundos")
print(f"Número de imagens de teste: {n_imagens_teste}")
print(f"Tempo médio por imagem: {tempo_inferencia_v5 / n_imagens_teste:.4f} segundos")

# Salva variável para uso na comparação final
tempo_medio_inferencia_v5 = tempo_inferencia_v5 / n_imagens_teste

In [ ]:
import glob
from IPython.display import Image as IPyImage, display

resultado_dir_v5 = 'yolov5/runs/detect/teste_yolov5'

imagens_resultado = (
    glob.glob(f'{resultado_dir_v5}/*.jpg') +
    glob.glob(f'{resultado_dir_v5}/*.jpeg') +
    glob.glob(f'{resultado_dir_v5}/*.png')
)

if imagens_resultado:
    print(f"Exibindo {len(imagens_resultado)} imagem(ns) processada(s) pelo YOLOv5:\n")
    for img_path in imagens_resultado:
        print(f"→ {os.path.basename(img_path)}")
        display(IPyImage(filename=img_path, width=640))
else:
    print(f"Nenhuma imagem encontrada em: {resultado_dir_v5}")

### Resultados do YOLOv5 — Resumo

| Classe      | Imagens | Instâncias | Precision | Recall | mAP50 | mAP50-95 |
|-------------|---------|------------|-----------|--------|-------|----------|
| all         | 20      | 86         | 0.818     | 0.847  | 0.859 | 0.523    |
| maquinarios | 20      | 12         | 1.000     | 0.883  | 0.916 | 0.585    |
| animais     | 20      | 74         | 0.635     | 0.811  | 0.801 | 0.462    |

**Tempo de treinamento (50 épocas):** 638.24 segundos (10.64 minutos)  
**Tempo médio de inferência por imagem:** preencher após rodar a célula de teste